## Celula 1 -- Instalacao

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.55.2
!pip install --no-deps trl==0.22.2
!pip install scikit-learn
!pip install neo4j
print("Done")

## Celula 3 -- Carregar Grafo

In [2]:
from neo4j import GraphDatabase
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

# Recupera o valor de cada secret
neo4j_uri      = user_secrets.get_secret("NEO4J_URI")
neo4j_user     = user_secrets.get_secret("NEO4J_USER")
neo4j_password = user_secrets.get_secret("NEO4J_PASSWORD")

driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))

def run_query(query, parameters=None):
    with driver.session() as session:
        result = session.run(query, parameters)
        return [record.data() for record in result]

## Celula 4 -- Mostra Grafo

In [16]:
def get_all_topics(driver):
    query = """
    MATCH (t:Topic)
    RETURN t.title_pt AS title
    """
    with driver.session() as session:
        result = session.run(query)
        return [record["title"] for record in result]

topic_list = get_all_topics(driver)
print("Total de tópicos:", len(topic_list))
#print("Primeiros 5:", topic_list[:5])

Total de tópicos: 48


## Celula 2 -- Carregar modelo e QLoRA

In [4]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 4096
SEED = 42

# Carrega o modelo base -- get_peft_model adiciona os adaptadores LoRA treinaveis
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit",
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = True,
    dtype          = None,
)

#model = FastLanguageModel.get_peft_model(
    #model,
    #r                        = 32,
    #target_modules           = ["q_proj", "k_proj", "v_proj", "o_proj",
                                #"gate_proj", "up_proj", "down_proj"],
    #lora_alpha               = 32,
    #lora_dropout             = 0,
    #bias                     = "none",
    #use_gradient_checkpointing = "unsloth",
    #random_state             = SEED,
    #use_rslora               = False,
    #loftq_config             = None,
#)

# Modo de inferencia do Unsloth -- geracao ~2x mais rapida
FastLanguageModel.for_inference(model)

# O Qwen3-4B-Instruct-2507 ja traz o chat template correto no tokenizer,
# portanto nao chamamos get_chat_template (era redundante).

# Verificacao de parametros treinaveis
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Parametros treinaveis: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth_zoo/__init__.py:527: UserWarning: Unsloth fused-forward install skipped: requires transformers >= 4.56.0.
  _install_fused_forward()


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.2: Fast Qwen3 patching. Transformers: 4.55.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
total weights      : 3.981 GiB
no_split classes   : ['Qwen3DecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.359 GiB
activation reserve : 10.817 GiB requested
tied to head       : ['model.embed_tokens']
  cuda:0  budget  12.98 GiB  weights  2.136 GiB  free 10.843 GiB  reserve 10.817 GiB
  cuda:1  budget  13.00 GiB  weights  1.846 GiB  free 11.151 GiB  reserve 10.647 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_memory lowered the budget on cu

## Celula 12 -- Prompts

In [5]:
from datetime import datetime
def ask_qwen(user_prompt: str, system_prompt: str = None, max_new_tokens: int = 256) -> str:
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Sem truncation: cortar pela direita apagaria as INSTRUCOES finais e o
    # marcador <|im_start|>assistant, fazendo o modelo continuar o enunciado
    # em vez de responder.
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    input_len = inputs["input_ids"].shape[1]
    if input_len + max_new_tokens > MAX_SEQ_LENGTH:
        raise ValueError(
            f"Prompt de {input_len} tokens + {max_new_tokens} novos tokens "
            f"excede MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}."
        )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,        # agora ajustavel
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=[tokenizer.convert_tokens_to_ids("<|im_end|>"), tokenizer.eos_token_id],
        )

    new_tokens = outputs[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
print("Terminado em: " + timestamp)

Terminado em: 20260904_044053


## Celula 14 -- Execução de Prompts

In [6]:
# Texto do problema (copiado da sua descrição)
problem_text = """time limit per test: 1 second
memory limit per test: 256 megabytes
You are given two integers n and k.
Construct a binary string s of length n, such that both of the following conditions hold:
The absolute difference between the number of characters 0 and the number of characters 1 in s is at most 1.
There are exactly k pairs of adjacent equal characters in s. Formally, there are exactly k indices i(1≤i≤n−1) satisfying si=si+1.
Or determine that no such string exists.
A binary string is a string where each character is either 0 or 1.

Input: Each test contains multiple test cases. The first line contains the number of test cases t (1≤t≤1000). The description of the test cases follows. The only line of each test case contains two integers n and k (2≤n≤2⋅10^5, 0≤k≤n−1). It is guaranteed that the sum of n over all test cases does not exceed 2⋅10^5.

Output: For each test case, output a binary string s of length n — the string you constructed. Print −1 if such a string does not exist. If there are multiple answers, you may output any of them.

Example
Input
8
5 2
4 3
6 1
5 0
7 3
4 2
3 2
7 4
Output
01110
-1
101001
01010
0100011
0011
-1
0111000"""

# # Definição dos prompts
# prompt_code = f"""Write a Python solution for the following competitive programming problem. 
# Return only the code, no explanation.

# {problem_text}"""

# prompt_pseudocode = f"""Write a pseudocode solution for the following competitive programming problem. 
# Return only the pseudocode, no explanation.

# {problem_text}"""

# prompt_semantic = f"""Rewrite the following competitive programming problem in plain English, 
# highlighting the key algorithmic concepts and what the task asks for. 
# Do not solve it, just explain the problem clearly.

# {problem_text}"""

# prompt_topics = f"""List the algorithmic topics that a student needs to know to solve this problem. 
# Return only a comma-separated list of topics, no explanation.

# {problem_text}"""

# # Dicionário de prompts
# prompts = {
#     "Código Python": prompt_code,
#     "Pseudocódigo": prompt_pseudocode,
#     "Transformação semântica": prompt_semantic,
#     "Tópicos": prompt_topics,
# }

# # Limites de tokens por tipo de resposta
# max_tokens = {
#     "Código Python": 1024,
#     "Pseudocódigo": 1024,
#     "Transformação semântica": 200,
#     "Tópicos": 200,
# }

# # Execução
# results = {}
# for label, prompt in prompts.items():
#     print(f"=== {label} ===")
#     answer = ask_qwen(prompt, max_new_tokens=max_tokens[label])
#     results[label] = answer
#     print(answer)
#     print("\n" + "-"*60 + "\n")

In [17]:
# Lista de tópicos em uma única linha, separada por vírgulas
topic_list_str = ', '.join(topic_list)
print(topic_list_str)

prompt_restrito = f"""Sua tarefa é selecionar tópicos de uma lista fornecida.

LISTA DE TÓPICOS DISPONÍVEIS:
{topic_list_str}

PROBLEMA:
{problem_text}

INSTRUÇÕES:
- Escolha APENAS os tópicos da LISTA DE TÓPICOS DISPONÍVEIS que são necessários para resolver o problema.
- Retorne exatamente os nomes dos tópicos selecionados, separados por vírgula.
- Não adicione explicações, não resolva o problema, não repita o enunciado.
- Se nenhum tópico for relevante, retorne "nenhum".

Resposta:
"""

selected_topics_text = ask_qwen(prompt_restrito, max_new_tokens=256)
print("Tópicos selecionados pelo Qwen:")
print(selected_topics_text)

Variáveis e tipos de dados, Estruturas condicionais (if/else), Laços de repetição (for/while), Funções e escopo, Arrays / vetores, Manipulação básica de strings, Recursão, Notação assintótica (Big-O), Análise de recorrências, Análise probabilística e aleatorização, Análise amortizada, Divisão e conquista, Programação dinâmica, Algoritmos gulosos, Busca linear e binária, Insertion sort, Merge sort, Heap e heapsort, Quicksort, Ordenação em tempo linear (counting/radix/bucket), Estatísticas de ordem / seleção, Listas encadeadas, pilhas e filas, Árvores binárias de busca, Árvores balanceadas (rubro-negra), Árvores B, Tabelas hash, Estruturas de dados aumentadas, Heaps de Fibonacci, Árvores de van Emde Boas, Conjuntos disjuntos (Union-Find), Representação de grafos, Busca em largura (BFS), Busca em profundidade (DFS), Ordenação topológica, Componentes fortemente conexos, Árvore geradora mínima, Caminhos mínimos de origem única, Caminhos mínimos entre todos os pares, Fluxo máximo, Operações 

## Celula 15 -- Validação dos tópicos selecionados

In [20]:
# ==========================================================
# VALIDAÇÃO DOS TÓPICOS RETORNADOS PELO MODELO
# Filtra a resposta contra topic_list e mede a taxa de aluci-
# nação (tópicos inventados ou fora da lista fornecida).
# ==========================================================

def validar_topicos(resposta: str, topic_list: list):
    """Separa a resposta do modelo em tópicos válidos (presentes em
    topic_list) e alucinados (inventados ou com grafia divergente)."""
    if resposta.strip().lower() == "nenhum":
        return [], [], 0.0

    # normalização: casa caixa e espaços sem alterar o nome oficial
    canon = {t.strip().lower(): t for t in topic_list}

    brutos = [t.strip() for t in resposta.split(",") if t.strip()]
    validos, alucinados = [], []
    for t in brutos:
        oficial = canon.get(t.lower())
        if oficial is not None:
            if oficial not in validos:      # descarta duplicatas
                validos.append(oficial)
        else:
            alucinados.append(t)

    taxa = len(alucinados) / len(brutos) if brutos else 0.0
    return validos, alucinados, taxa


topicos_validos, topicos_alucinados, taxa_alucinacao = validar_topicos(
    selected_topics_text, topic_list
)

print(f"Válidos ({len(topicos_validos)}):")
for t in topicos_validos:
    print("  -", t)

print(f"\nAlucinados ({len(topicos_alucinados)}):")
for t in topicos_alucinados:
    print("  -", t)

print(f"\nTaxa de alucinação: {taxa_alucinacao:.1%}")

Válidos (5):
  - Variáveis e tipos de dados
  - Estruturas condicionais (if/else)
  - Laços de repetição (for/while)
  - Manipulação básica de strings
  - Funções e escopo

Alucinados (0):

Taxa de alucinação: 0.0%


## Celula 16 -- Percurso no grafo a partir dos tópicos escolhidos

In [21]:
# ==========================================================
# PERCURSO NO GRAFO
# Parte dos tópicos escolhidos pelo Qwen (topicos_validos) e
# percorre as arestas PREREQUISITE_OF no sentido inverso para
# recuperar toda a cadeia de pré-requisitos, montando uma
# ordem de estudo.
#
# Estrutura do grafo (48 nós, 79 arestas, DAG de profundidade 9):
#   (:Topic {title_pt, code, category, book_ref})
#   (pre:Topic)-[:PREREQUISITE_OF]->(dependente:Topic)
# ==========================================================
from collections import defaultdict


def buscar_prerequisitos(titulos):
    """Pré-requisitos diretos e transitivos dos tópicos selecionados.
    `distancia` = menor número de saltos até algum tópico selecionado."""
    query = """
    MATCH (alvo:Topic)
    WHERE alvo.title_pt IN $titulos
    MATCH caminho = (pre:Topic)-[:PREREQUISITE_OF*1..]->(alvo)
    WHERE NOT pre.title_pt IN $titulos
    RETURN pre.title_pt AS titulo,
           pre.code     AS code,
           pre.category AS categoria,
           pre.book_ref AS book_ref,
           min(length(caminho))            AS distancia,
           collect(DISTINCT alvo.title_pt) AS requerido_por
    ORDER BY distancia, titulo
    """
    return run_query(query, {"titulos": titulos})


def buscar_metadados(titulos):
    """Propriedades dos tópicos, na ordem em que forem pedidos."""
    query = """
    MATCH (t:Topic)
    WHERE t.title_pt IN $titulos
    RETURN t.title_pt AS titulo,
           t.code     AS code,
           t.category AS categoria,
           t.book_ref AS book_ref
    """
    por_titulo = {r["titulo"]: r for r in run_query(query, {"titulos": titulos})}
    return [por_titulo[t] for t in titulos if t in por_titulo]


def buscar_arestas(titulos):
    """Arestas PREREQUISITE_OF internas ao subgrafo induzido."""
    query = """
    MATCH (a:Topic)-[:PREREQUISITE_OF]->(b:Topic)
    WHERE a.title_pt IN $titulos AND b.title_pt IN $titulos
    RETURN a.title_pt AS de, b.title_pt AS para
    ORDER BY de, para
    """
    return run_query(query, {"titulos": titulos})


def ordem_topologica(nos, arestas):
    """Kahn com desempate alfabético -> ordem de estudo determinística."""
    grau = {n: 0 for n in nos}
    suc = defaultdict(list)
    for a in arestas:
        if a["de"] in grau and a["para"] in grau:
            suc[a["de"]].append(a["para"])
            grau[a["para"]] += 1

    fila, ordem = sorted(n for n in nos if grau[n] == 0), []
    while fila:
        n = fila.pop(0)
        ordem.append(n)
        liberados = []
        for v in suc[n]:
            grau[v] -= 1
            if grau[v] == 0:
                liberados.append(v)
        fila = sorted(fila + liberados)

    if len(ordem) != len(nos):      # so aconteceria se o grafo tivesse ciclo
        print("AVISO: ciclo detectado no subgrafo; tópicos restantes anexados ao fim.")
        ordem += sorted(set(nos) - set(ordem))
    return ordem


def fmt_ref(book_ref):
    return ", ".join(book_ref) if book_ref else "--"


# ---------------------- execução ----------------------
selecionados = topicos_validos          # vem da Celula 15 (validação)

if not selecionados:
    raise ValueError("Nenhum tópico válido foi selecionado pelo modelo; "
                     "nada a percorrer no grafo.")

topicos_prerequisitos = buscar_prerequisitos(selecionados)
subgrafo_nos = selecionados + [p["titulo"] for p in topicos_prerequisitos]
subgrafo_arestas = buscar_arestas(subgrafo_nos)
ordem_estudo = ordem_topologica(subgrafo_nos, subgrafo_arestas)
meta = {m["titulo"]: m for m in buscar_metadados(subgrafo_nos)}

print("=" * 78)
print(f"TÓPICOS SELECIONADOS PELO MODELO ({len(selecionados)})")
print("=" * 78)
for t in selecionados:
    m = meta[t]
    print(f"  {t}\n      categoria: {m['categoria']} | CLRS: {fmt_ref(m['book_ref'])}")

print()
print("=" * 78)
print(f"PRÉ-REQUISITOS RECUPERADOS DO GRAFO ({len(topicos_prerequisitos)})")
print("=" * 78)
for p in topicos_prerequisitos:
    print(f"  [dist {p['distancia']}] {p['titulo']}")
    print(f"      categoria: {p['categoria']} | CLRS: {fmt_ref(p['book_ref'])}")
    print(f"      requerido por: {', '.join(sorted(p['requerido_por']))}")

print()
print("=" * 78)
print(f"ORDEM DE ESTUDO -- ordenação topológica ({len(ordem_estudo)} tópicos)")
print("=" * 78)
for i, t in enumerate(ordem_estudo, 1):
    marca = "*" if t in selecionados else " "
    print(f"  {i:2d}. {marca} {t}  [{fmt_ref(meta[t]['book_ref'])}]")
print("\n  (* = escolhido pelo modelo; demais = pré-requisitos vindos do grafo)")

print()
print("=" * 78)
print("ARESTAS DO SUBGRAFO INDUZIDO")
print("=" * 78)
for a in subgrafo_arestas:
    print(f"  {a['de']} --PREREQUISITE_OF--> {a['para']}")

capitulos = sorted({c for t in ordem_estudo for c in meta[t]["book_ref"]})
print(f"\nCapítulos CLRS envolvidos ({len(capitulos)}): {', '.join(capitulos)}")

TÓPICOS SELECIONADOS PELO MODELO (5)
  Variáveis e tipos de dados
      categoria: fundamentos_programacao | CLRS: --
  Estruturas condicionais (if/else)
      categoria: fundamentos_programacao | CLRS: --
  Laços de repetição (for/while)
      categoria: fundamentos_programacao | CLRS: --
  Manipulação básica de strings
      categoria: fundamentos_programacao | CLRS: --
  Funções e escopo
      categoria: fundamentos_programacao | CLRS: --

PRÉ-REQUISITOS RECUPERADOS DO GRAFO (1)
  [dist 1] Arrays / vetores
      categoria: fundamentos_programacao | CLRS: --
      requerido por: Manipulação básica de strings

ORDEM DE ESTUDO -- ordenação topológica (6 tópicos)
   1. * Variáveis e tipos de dados  [--]
   2. * Estruturas condicionais (if/else)  [--]
   3. * Laços de repetição (for/while)  [--]
   4.   Arrays / vetores  [--]
   5. * Funções e escopo  [--]
   6. * Manipulação básica de strings  [--]

  (* = escolhido pelo modelo; demais = pré-requisitos vindos do grafo)

ARESTAS DO SUBGR

## Celula 17 -- Painel de juízes: configuração (Groq / Gemini / Mistral)

Adiciona votação por consenso **depois** da geração da trilha. Secrets: `GROQ_API_KEY`, `GEMINI_API_KEY` e, opcionalmente, `MISTRAL_API_KEY`. Para incluir outro modelo, basta acrescentar um `JudgeConfig` (qualquer API compatível com OpenAI).

In [ ]:
# ==========================================================
# PAINEL DE JUIZES -- configuracao
# Todos os provedores abaixo expoem API compativel com OpenAI
# (POST {base_url}/chat/completions), entao um unico cliente
# baseado em `requests` atende a todos. Para adicionar um juiz
# novo, basta incluir um JudgeConfig em JUDGES_CANDIDATOS.
#
# Secrets esperados (Kaggle Secrets ou variavel de ambiente):
#   GROQ_API_KEY, GEMINI_API_KEY e (opcional) MISTRAL_API_KEY
# ==========================================================
import os, re, json, time, random, hashlib
from collections import Counter
from dataclasses import dataclass, field
from concurrent.futures import ThreadPoolExecutor
import requests


def get_secret(nome):
    """Kaggle Secrets -> variavel de ambiente -> None."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(nome)
    except Exception:
        return os.environ.get(nome)


@dataclass
class JudgeConfig:
    nome: str
    base_url: str
    model: str
    secret_name: str
    extra_body: dict = field(default_factory=dict)  # params especificos do provedor
    max_tokens: int = 4096    # em modelos "thinking" inclui os tokens de raciocinio
    timeout: int = 120


# Familias diferentes de proposito: o gerador e Qwen, entao evitamos juizes
# Qwen (vies de autopreferencia e erros correlacionados).
# CONFIRME os IDs no console de cada provedor -- eles mudam com frequencia.
JUDGES_CANDIDATOS = [
    # Groq (free: ~30 RPM, 1K req/dia, ~8K tokens/min -> o gargalo e o TPM)
    JudgeConfig(
        nome="groq/gpt-oss-120b",
        base_url="https://api.groq.com/openai/v1",
        model="openai/gpt-oss-120b",
        secret_name="GROQ_API_KEY",
        extra_body={"reasoning_effort": "medium"},
        max_tokens=3000,
    ),
    # Gemini (free apenas para Flash/Flash-Lite). O alias "-latest" aponta
    # para o Flash mais recente; troque por um ID fixo para reprodutibilidade.
    JudgeConfig(
        nome="gemini/flash",
        base_url="https://generativelanguage.googleapis.com/v1beta/openai",
        model="gemini-flash-latest",
        secret_name="GEMINI_API_KEY",
    ),
    # Mistral (plano gratuito "Experiment"; exige aceitar uso dos dados p/ treino)
    JudgeConfig(
        nome="mistral/medium",
        base_url="https://api.mistral.ai/v1",
        model="mistral-medium-latest",
        secret_name="MISTRAL_API_KEY",
    ),
]

JUDGES = []   # lista de (JudgeConfig, api_key) -- so entram juizes com chave
for _cfg in JUDGES_CANDIDATOS:
    _key = get_secret(_cfg.secret_name)
    if _key:
        JUDGES.append((_cfg, _key))
    else:
        print(f"[aviso] secret {_cfg.secret_name} ausente -> juiz '{_cfg.nome}' desativado")

print(f"Juizes ativos ({len(JUDGES)}): {[c.nome for c, _ in JUDGES]}")
if len(JUDGES) < 3:
    print("[aviso] com menos de 3 juizes, empates viram INCONCLUSIVO "
          "(nenhuma correcao e disparada e nada entra no banco de exemplos).")


## Celula 18 -- Painel de juízes: prompt, chamada e votação por maioria

Veredito por juiz: `CORRETA` / `INCORRETA` + justificativa + tópicos faltando/desnecessários (validados contra o grafo). Consenso: maioria entre votos válidos; empate ou quórum insuficiente = `INCONCLUSIVO`.

In [ ]:
# ==========================================================
# PAINEL DE JUIZES -- cliente, prompt e votacao por consenso
# ==========================================================

def _extrair_json(texto):
    """Extrai o primeiro objeto JSON da resposta (tolera <think>, ```json e texto extra)."""
    texto = re.sub(r"<think>.*?</think>", "", texto or "", flags=re.S)
    texto = texto.replace("```json", "").replace("```", "")
    ini = texto.find("{")
    if ini == -1:
        raise ValueError("resposta sem JSON")
    prof, em_str, esc = 0, False, False
    for i in range(ini, len(texto)):
        c = texto[i]
        if em_str:
            if esc:
                esc = False
            elif c == "\\":
                esc = True
            elif c == '"':
                em_str = False
        elif c == '"':
            em_str = True
        elif c == "{":
            prof += 1
        elif c == "}":
            prof -= 1
            if prof == 0:
                return json.loads(texto[ini:i + 1])
    raise ValueError("JSON incompleto (resposta truncada?)")


def _post_chat(cfg, api_key, messages, usar_opcionais=True, max_tentativas=4):
    """Chama /chat/completions com retry em 429/5xx (respeitando retry-after).
    Se o provedor recusar parametros opcionais (HTTP 400), repete sem eles."""
    url = cfg.base_url.rstrip("/") + "/chat/completions"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    body = {"model": cfg.model, "messages": messages,
            "temperature": 0, "max_tokens": cfg.max_tokens}
    if usar_opcionais:
        body["response_format"] = {"type": "json_object"}
        body.update(cfg.extra_body)

    for tentativa in range(max_tentativas):
        r = requests.post(url, headers=headers, json=body, timeout=cfg.timeout)
        if r.status_code == 200:
            return r.json()["choices"][0]["message"]["content"] or ""
        if r.status_code == 400 and usar_opcionais:
            return _post_chat(cfg, api_key, messages, usar_opcionais=False,
                              max_tentativas=max(max_tentativas - tentativa, 1))
        if r.status_code in (429, 500, 502, 503, 504):
            try:
                espera = float(r.headers.get("retry-after"))
            except (TypeError, ValueError):
                espera = 2 ** (tentativa + 2)
            if espera > 90:   # cota diaria esgotada, nao adianta esperar
                raise RuntimeError(f"cota esgotada (retry-after={espera:.0f}s)")
            time.sleep(espera + random.random())
            continue
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:300]}")
    raise RuntimeError("esgotou as tentativas (rate limit / indisponibilidade)")


def _normalizar_veredito(v):
    v = str(v or "").strip().upper()
    if v.startswith("INCORRET"):      # checar antes: "INCORRETA" nao comeca com "CORRET"
        return "INCORRETA"
    if v.startswith("CORRET"):
        return "CORRETA"
    return None


def _filtrar_topicos(itens):
    """Mantem so nomes que existem em topic_list (juizes tambem alucinam)."""
    canon = {t.strip().lower(): t for t in topic_list}
    if isinstance(itens, str):
        itens = [itens]
    saida = []
    for t in itens or []:
        oficial = canon.get(str(t).strip().lower())
        if oficial and oficial not in saida:
            saida.append(oficial)
    return saida


PROMPT_JUIZ = """Você é um avaliador especialista em programação competitiva e em planejamento de estudos.

Um modelo recebeu um PROBLEMA e uma LISTA FECHADA de tópicos (nós de um grafo de conhecimento baseado em um livro-texto de algoritmos). Ele escolheu os tópicos necessários para resolver o problema; em seguida, o sistema adicionou automaticamente os pré-requisitos do grafo e ordenou tudo topologicamente, formando uma TRILHA DE APRENDIZADO.

PROBLEMA:
<<PROBLEMA>>

LISTA FECHADA DE TÓPICOS (único vocabulário permitido):
<<TOPICOS>>

TRILHA AVALIADA:
<<TRILHA>>

COMO AVALIAR:
1. Pense na solução esperada do problema e identifique quais tópicos da lista são realmente necessários para chegar a ela.
2. SUFICIÊNCIA: algum tópico essencial está ausente da trilha (nem escolhido, nem pré-requisito)?
3. NECESSIDADE: algum tópico escolhido é irrelevante para este problema, ou o escopo da trilha está claramente maior ou menor do que a dificuldade do problema exige?
4. Marque INCORRETA somente se conseguir apontar um erro concreto (tópico essencial faltando ou tópico claramente irrelevante). Diferenças de preferência ou tópicos "úteis, mas dispensáveis" NÃO tornam a trilha incorreta.

Responda APENAS com um objeto JSON, sem nenhum texto fora dele:
{"veredito": "CORRETA" ou "INCORRETA",
 "confianca": número entre 0 e 1,
 "justificativa": "explicação objetiva em até 5 frases, citando o erro concreto se INCORRETA",
 "topicos_faltando": ["nomes EXATOS da lista fechada que deveriam estar na trilha"],
 "topicos_desnecessarios": ["nomes EXATOS da lista fechada, apenas entre os escolhidos pelo modelo, que não deveriam estar"]}
"""


def montar_prompt_juiz(problem_text, trilha):
    return (PROMPT_JUIZ
            .replace("<<PROBLEMA>>", problem_text)
            .replace("<<TOPICOS>>", ", ".join(topic_list))
            .replace("<<TRILHA>>", trilha_para_texto(trilha)))


def _julgar_um(cfg, api_key, prompt):
    msgs = [
        {"role": "system", "content": "Você é um avaliador rigoroso e imparcial. Responda sempre e somente com JSON válido."},
        {"role": "user", "content": prompt},
    ]
    t0 = time.time()
    try:
        d = _extrair_json(_post_chat(cfg, api_key, msgs))
        veredito = _normalizar_veredito(d.get("veredito"))
        if veredito is None:
            raise ValueError(f"veredito invalido: {d.get('veredito')!r}")
        try:
            conf = min(max(float(d.get("confianca", 0.5)), 0.0), 1.0)
        except (TypeError, ValueError):
            conf = 0.5
        return {"juiz": cfg.nome, "veredito": veredito, "confianca": conf,
                "justificativa": str(d.get("justificativa", "")).strip(),
                "topicos_faltando": _filtrar_topicos(d.get("topicos_faltando")),
                "topicos_desnecessarios": _filtrar_topicos(d.get("topicos_desnecessarios")),
                "erro": None, "latencia_s": round(time.time() - t0, 1)}
    except Exception as e:
        return {"juiz": cfg.nome, "veredito": "ABSTENCAO", "confianca": 0.0,
                "justificativa": "", "topicos_faltando": [], "topicos_desnecessarios": [],
                "erro": f"{type(e).__name__}: {e}", "latencia_s": round(time.time() - t0, 1)}


def consolidar_votos(votos, quorum=2):
    """Maioria simples entre os votos validos.
    - menos de `quorum` votos validos ou empate -> INCONCLUSIVO
    - `vencedor` = voto da maioria com maior confianca (fonte da justificativa)"""
    validos = [v for v in votos if v["veredito"] in ("CORRETA", "INCORRETA")]
    contagem = Counter(v["veredito"] for v in validos)
    resultado = {"veredito": "INCONCLUSIVO", "contagem": dict(contagem),
                 "abstencoes": len(votos) - len(validos), "unanime": False,
                 "vencedor": None, "votos": votos}
    if len(validos) < quorum:
        return resultado
    ranking = contagem.most_common()
    if len(ranking) > 1 and ranking[0][1] == ranking[1][1]:
        return resultado
    veredito = ranking[0][0]
    maioria = [v for v in validos if v["veredito"] == veredito]
    resultado.update(veredito=veredito,
                     unanime=(len(maioria) == len(validos)),
                     vencedor=max(maioria, key=lambda v: v["confianca"]))
    return resultado


def julgar_trilha(problem_text, trilha, quorum=2):
    """Consulta todos os juizes em paralelo (provedores distintos -> cotas independentes)."""
    if not JUDGES:
        raise RuntimeError("Nenhum juiz ativo: configure ao menos uma API key (Celula 17).")
    prompt = montar_prompt_juiz(problem_text, trilha)
    with ThreadPoolExecutor(max_workers=len(JUDGES)) as ex:
        votos = list(ex.map(lambda jk: _julgar_um(jk[0], jk[1], prompt), JUDGES))
    return consolidar_votos(votos, quorum=quorum)


def imprimir_painel(painel, largura=220):
    print(f"  Painel: {painel['veredito']}  votos={painel['contagem']}  "
          f"abstencoes={painel['abstencoes']}  unanime={painel['unanime']}")
    for v in painel["votos"]:
        if v["veredito"] == "ABSTENCAO":
            print(f"    - {v['juiz']}: ABSTENCAO ({v['erro']})")
        else:
            print(f"    - {v['juiz']}: {v['veredito']} (conf {v['confianca']:.2f}) "
                  f"{v['justificativa'][:largura]}")


## Celula 19 -- Qwen multi-turno e construção da trilha

Encapsula o fluxo das Células 14-16 (selecionar → validar → percorrer o grafo) em funções reutilizáveis. Com o bloco de exemplos vazio, o prompt é idêntico ao `prompt_restrito` original.

In [ ]:
# ==========================================================
# QWEN (multi-turno) + CONSTRUCAO DA TRILHA
# Reaproveita as funcoes das Celulas 15 e 16 (validar_topicos,
# buscar_prerequisitos, buscar_arestas, ordem_topologica,
# buscar_metadados) sem altera-las.
# ==========================================================
MAX_NEW_SELECAO = 256


def contar_tokens(texto):
    return len(tokenizer(texto, add_special_tokens=False)["input_ids"])


def ask_qwen_messages(messages, max_new_tokens=MAX_NEW_SELECAO):
    """Igual a ask_qwen (Celula 12), mas recebe a lista de mensagens inteira.
    Necessario para o ciclo de correcao: [pedido, resposta anterior, critica]."""
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_len = inputs["input_ids"].shape[1]
    if input_len + max_new_tokens > MAX_SEQ_LENGTH:
        raise ValueError(f"Prompt de {input_len} tokens + {max_new_tokens} novos "
                         f"excede MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}.")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=[tokenizer.convert_tokens_to_ids("<|im_end|>"), tokenizer.eos_token_id],
        )
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


# Mesmo texto do prompt_restrito da Celula 14; com bloco de exemplos vazio o
# resultado e identico ao original.
PROMPT_SELECAO = """Sua tarefa é selecionar tópicos de uma lista fornecida.

LISTA DE TÓPICOS DISPONÍVEIS:
<<TOPICOS>>

<<EXEMPLOS>>PROBLEMA:
<<PROBLEMA>>

INSTRUÇÕES:
- Escolha APENAS os tópicos da LISTA DE TÓPICOS DISPONÍVEIS que são necessários para resolver o problema.
- Retorne exatamente os nomes dos tópicos selecionados, separados por vírgula.
- Não adicione explicações, não resolva o problema, não repita o enunciado.
- Se nenhum tópico for relevante, retorne "nenhum".

Resposta:
"""


def montar_prompt_selecao(problem_text, bloco_exemplos=""):
    exemplos = (bloco_exemplos.strip() + "\n\n") if bloco_exemplos else ""
    return (PROMPT_SELECAO
            .replace("<<TOPICOS>>", ", ".join(topic_list))
            .replace("<<EXEMPLOS>>", exemplos)
            .replace("<<PROBLEMA>>", problem_text))


def construir_trilha(resposta):
    """Resposta do Qwen -> validacao -> percurso no grafo -> trilha ordenada."""
    validos, alucinados, taxa = validar_topicos(resposta, topic_list)
    trilha = {"resposta": resposta, "selecionados": validos, "alucinados": alucinados,
              "taxa_alucinacao": taxa, "prerequisitos": [], "ordem": [], "arestas": [], "meta": {}}
    if not validos:
        return trilha
    prereqs = buscar_prerequisitos(validos)
    nos = validos + [p["titulo"] for p in prereqs]
    arestas = buscar_arestas(nos)
    trilha.update(prerequisitos=prereqs, arestas=arestas,
                  ordem=ordem_topologica(nos, arestas),
                  meta={m["titulo"]: m for m in buscar_metadados(nos)})
    return trilha


def trilha_para_texto(trilha):
    sel = trilha["selecionados"]
    pre = [t for t in trilha["ordem"] if t not in sel]
    linhas = ["Tópicos escolhidos pelo modelo:"] + [f"- {t}" for t in sel]
    linhas += ["", "Pré-requisitos adicionados automaticamente pelo grafo:"]
    linhas += [f"- {t}" for t in pre] or ["- (nenhum)"]
    linhas += ["", "Ordem de estudo resultante:"]
    linhas += [f"{i}. {t}" + (" (escolhido)" if t in sel else "")
               for i, t in enumerate(trilha["ordem"], 1)]
    return "\n".join(linhas)


## Celula 20 -- Pipeline 2: reforço de prompt (banco de exemplos)

Exemplos *Como fazer* (seleção correta) e *O que evitar* (seleção errada + erro apontado + correção), escolhidos por similaridade (TF-IDF) dentro de um orçamento de tokens. Ground truth humano tem prioridade sobre exemplos validados pelos juízes.

In [ ]:
# ==========================================================
# PIPELINE 2 -- REFORCO DE PROMPT
# Banco de exemplos "Como fazer" (selecao correta) e "O que evitar"
# (selecao errada + erro apontado + correcao). Os exemplos entram
# FIXOS no prompt do Qwen; os mais parecidos com o problema atual
# (TF-IDF) sao escolhidos, respeitando um orcamento de tokens.
# ==========================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

BANCO_PATH = "/kaggle/working/banco_exemplos.json"


def id_problema(texto):
    return hashlib.md5(" ".join(texto.split()).encode("utf-8")).hexdigest()[:12]


def _compacta(texto, n):
    texto = " ".join(texto.split())
    return texto if len(texto) <= n else texto[:n].rstrip() + "..."


class BancoExemplos:
    """Cada item: id, problema, tipo ('bom'|'ruim'), fonte ('ground_truth'|'juizes'),
    topicos (bom: selecao correta | ruim: -), resposta_errada, justificativa,
    topicos_faltando, topicos_desnecessarios, topicos_corrigidos, confianca."""

    BONUS_GROUND_TRUTH = 0.15   # exemplo verificado por humanos ganha prioridade

    def __init__(self, path=BANCO_PATH):
        self.path, self.itens = path, []
        if os.path.exists(path):
            with open(path, encoding="utf-8") as f:
                self.itens = json.load(f)

    def _salvar(self):
        with open(self.path, "w", encoding="utf-8") as f:
            json.dump(self.itens, f, ensure_ascii=False, indent=2)

    def adicionar(self, item):
        antigo = next((i for i in self.itens
                       if i["id"] == item["id"] and i["tipo"] == item["tipo"]), None)
        if antigo is not None:
            if antigo["fonte"] == "ground_truth" and item["fonte"] != "ground_truth":
                return False                      # nunca sobrescreve verdade humana
            self.itens.remove(antigo)
        self.itens.append(item)
        self._salvar()
        return True

    def resumo(self):
        c = Counter((i["fonte"], i["tipo"]) for i in self.itens)
        return {f"{f}/{t}": n for (f, t), n in sorted(c.items())} or "(vazio)"

    # ---------------- selecao ----------------
    def selecionar(self, problem_text, k_bom=2, k_ruim=1):
        pid = id_problema(problem_text)
        cand = [i for i in self.itens if i["id"] != pid]   # leave-one-out
        if not cand:
            return [], []
        vet = TfidfVectorizer(stop_words="english", sublinear_tf=True)
        vet.fit([problem_text] + [c["problema"] for c in cand])
        sims = cosine_similarity(vet.transform([problem_text]),
                                 vet.transform([c["problema"] for c in cand]))[0]
        pontuados = []
        for c, s in zip(cand, sims):
            bonus = self.BONUS_GROUND_TRUTH if c["fonte"] == "ground_truth" else 0.0
            pontuados.append((s + bonus, c))
        pontuados.sort(key=lambda x: -x[0])

        # um item por problema (o de maior pontuacao)
        vistos, unicos = set(), []
        for _, c in pontuados:
            if c["id"] not in vistos:
                vistos.add(c["id"]); unicos.append(c)
        bons = [c for c in unicos if c["tipo"] == "bom"][:k_bom]
        ruins = [c for c in unicos if c["tipo"] == "ruim"][:k_ruim]
        return bons, ruins

    def construir_bloco_exemplos(self, problem_text, max_tokens, k_bom=2, k_ruim=1,
                                 max_chars_problema=700):
        bons, ruins = self.selecionar(problem_text, k_bom, k_ruim)
        intercalados = []
        for i in range(max(len(bons), len(ruins))):      # bom, ruim, bom, ...
            if i < len(bons):  intercalados.append(bons[i])
            if i < len(ruins): intercalados.append(ruins[i])
        escolhidos = []
        for it in intercalados:                           # enche ate o orcamento
            tentativa = escolhidos + [it]
            if contar_tokens(renderizar_exemplos(tentativa, max_chars_problema)) <= max_tokens:
                escolhidos = tentativa
        return renderizar_exemplos(escolhidos, max_chars_problema) if escolhidos else ""


def renderizar_exemplos(itens, max_chars_problema=700):
    if not itens:
        return ""
    linhas = ["EXEMPLOS DE REFERÊNCIA (já validados). Aprenda o padrão de raciocínio, "
              "mas NÃO copie os tópicos: escolha com base no PROBLEMA ATUAL, abaixo.", ""]
    n_bom = n_ruim = 0
    for it in itens:
        if it["tipo"] == "bom":
            n_bom += 1
            linhas += [f"[COMO FAZER {n_bom}]",
                       f"Problema: {_compacta(it['problema'], max_chars_problema)}",
                       f"Tópicos corretos: {', '.join(it['topicos'])}", ""]
        else:
            n_ruim += 1
            linhas += [f"[O QUE EVITAR {n_ruim}]",
                       f"Problema: {_compacta(it['problema'], max_chars_problema)}",
                       f"Seleção ERRADA: {_compacta(it['resposta_errada'], 300)}",
                       f"Erro apontado: {_compacta(it['justificativa'], 400)}"]
            if it.get("topicos_faltando"):
                linhas.append(f"Faltavam: {', '.join(it['topicos_faltando'])}")
            if it.get("topicos_desnecessarios"):
                linhas.append(f"Eram desnecessários: {', '.join(it['topicos_desnecessarios'])}")
            if it.get("topicos_corrigidos"):
                linhas.append(f"Seleção correta: {', '.join(it['topicos_corrigidos'])}")
            linhas.append("")
    return "\n".join(linhas).rstrip()


def registrar_resultado(banco, problem_text, tentativas, exigir_unanimidade=True):
    """Promove o resultado de uma execucao a exemplo do banco.
    So entra o que o painel decidiu (sem INCONCLUSIVO) e, por padrao, so por
    unanimidade -- um falso positivo dos juizes viraria 'verdade' fixa no prompt."""
    primeira = tentativas[0]
    painel = primeira["painel"]
    if painel["veredito"] not in ("CORRETA", "INCORRETA"):
        return None
    if exigir_unanimidade and not painel["unanime"]:
        return None
    v = painel["vencedor"]
    base = {"id": id_problema(problem_text), "problema": problem_text,
            "fonte": "juizes", "confianca": v["confianca"]}
    if painel["veredito"] == "CORRETA":
        item = {**base, "tipo": "bom", "topicos": primeira["trilha"]["selecionados"]}
    else:
        ultima = tentativas[-1]
        corrigidos = (ultima["trilha"]["selecionados"]
                      if len(tentativas) > 1 and ultima["painel"]["veredito"] == "CORRETA"
                      else None)
        item = {**base, "tipo": "ruim",
                "resposta_errada": primeira["resposta"],
                "justificativa": v["justificativa"],
                "topicos_faltando": v["topicos_faltando"],
                "topicos_desnecessarios": v["topicos_desnecessarios"],
                "topicos_corrigidos": corrigidos}
    return item if banco.adicionar(item) else None


## Celula 21 -- Pipeline 1: auto-correção (critique-correction loop)

Qwen gera → juízes votam → se `INCORRETA`, a justificativa do juiz vencedor volta ao Qwen na mesma conversa → nova trilha → novo julgamento.

In [ ]:
# ==========================================================
# PIPELINE 1 -- AUTO-CORRECAO (Critique-Correction Loop)
# Qwen gera -> painel vota -> se INCORRETA, a justificativa do
# juiz vencedor volta ao Qwen (mesma conversa) -> nova trilha ->
# novo julgamento, ate CORRETA ou `max_rodadas` correcoes.
# Se um banco de exemplos for passado, os exemplos entram no
# prompt inicial (Pipeline 2).
# ==========================================================
RESERVA_POR_RODADA = 600   # tokens reservados p/ resposta anterior + critica


def _painel_local(justificativa):
    """Veredito sem chamar API: o proprio validador ja sabe que a selecao e inutilizavel."""
    voto = {"juiz": "validador-local", "veredito": "INCORRETA", "confianca": 1.0,
            "justificativa": justificativa, "topicos_faltando": [],
            "topicos_desnecessarios": [], "erro": None}
    return {"veredito": "INCORRETA", "contagem": {"INCORRETA": 1}, "abstencoes": 0,
            "unanime": True, "vencedor": voto, "votos": [voto], "local": True}


def montar_critica(painel, trilha):
    v = painel["vencedor"]
    partes = [f"Os juízes apontaram o seguinte erro na sua trilha: {v['justificativa']}"]
    if v["topicos_faltando"]:
        partes.append("Tópicos que podem estar faltando: " + ", ".join(v["topicos_faltando"]) + ".")
    if v["topicos_desnecessarios"]:
        partes.append("Tópicos que podem ser desnecessários: " + ", ".join(v["topicos_desnecessarios"]) + ".")
    if trilha["alucinados"]:
        partes.append("Os itens a seguir NÃO existem na lista e devem ser descartados: "
                      + ", ".join(trilha["alucinados"]) + ".")
    partes.append("Corrija-a mantendo a complexidade esperada (sem incluir tópicos desnecessários "
                  "e sem omitir os essenciais). Responda novamente apenas com os nomes exatos dos "
                  "tópicos da LISTA DE TÓPICOS DISPONÍVEIS, separados por vírgula, sem explicações.")
    return "\n".join(partes)


def pipeline_autocorrecao(problem_text, max_rodadas=2, banco=None, registrar=True,
                          k_bom=2, k_ruim=1, quorum=2, verbose=True):
    # ---- prompt inicial (com exemplos fixos, se houver banco) ----
    bloco = ""
    if banco is not None and banco.itens:
        base = montar_prompt_selecao(problem_text)
        orcamento = (MAX_SEQ_LENGTH - MAX_NEW_SELECAO - contar_tokens(base)
                     - RESERVA_POR_RODADA * max_rodadas)
        if orcamento > 200:
            bloco = banco.construir_bloco_exemplos(problem_text, min(orcamento, 1500), k_bom, k_ruim)
    mensagens = [{"role": "user", "content": montar_prompt_selecao(problem_text, bloco)}]
    if verbose:
        print(f"[exemplos no prompt: {'sim' if bloco else 'nao'}]")

    tentativas, status = [], "nao_convergiu"
    for rodada in range(max_rodadas + 1):
        try:
            resposta = ask_qwen_messages(mensagens)
        except ValueError as e:                     # estourou o contexto
            status = "contexto_excedido"
            if verbose: print(f"  {e}")
            break
        trilha = construir_trilha(resposta)

        if tentativas and set(trilha["selecionados"]) == set(tentativas[-1]["trilha"]["selecionados"]):
            # o Qwen devolveu a mesma selecao: julgar de novo so gastaria cota
            tentativas.append({"rodada": rodada, "resposta": resposta, "trilha": trilha,
                               "painel": tentativas[-1]["painel"], "reaproveitado": True})
            status = "sem_progresso"
            if verbose: print(f"[rodada {rodada}] selecao inalterada -> encerrando")
            break

        if not trilha["selecionados"]:
            motivo = ("Nenhum dos itens retornados existe na lista de tópicos: "
                      + ", ".join(trilha["alucinados"]) + "." if trilha["alucinados"]
                      else "Nenhum tópico foi selecionado, mas todo problema exige algum conhecimento da lista.")
            painel = _painel_local(motivo)
        else:
            painel = julgar_trilha(problem_text, trilha, quorum=quorum)

        tentativas.append({"rodada": rodada, "resposta": resposta, "trilha": trilha, "painel": painel})
        if verbose:
            print(f"[rodada {rodada}] selecao: {trilha['selecionados']}  alucinados: {trilha['alucinados']}")
            imprimir_painel(painel)

        if painel["veredito"] == "CORRETA":
            status = "aprovada_direto" if rodada == 0 else "corrigida"
            break
        if painel["veredito"] == "INCONCLUSIVO":
            status = "inconclusiva"
            break
        if rodada == max_rodadas:
            break                                   # INCORRETA e sem correcoes restantes
        mensagens += [{"role": "assistant", "content": resposta},
                      {"role": "user", "content": montar_critica(painel, trilha)}]

    resultado = {"problema": problem_text, "status": status, "tentativas": tentativas,
                 "trilha_final": tentativas[-1]["trilha"] if tentativas else None,
                 "painel_final": tentativas[-1]["painel"] if tentativas else None,
                 "registrado_no_banco": None}
    if banco is not None and registrar and tentativas:
        resultado["registrado_no_banco"] = registrar_resultado(banco, problem_text, tentativas)
        if verbose:
            r = resultado["registrado_no_banco"]
            print(f"[banco] {'novo exemplo: ' + r['tipo'] if r else 'nada registrado'}  -> {banco.resumo()}")
    return resultado


def imprimir_resultado(res):
    print("=" * 78)
    print(f"STATUS: {res['status']}  |  rodadas: {len(res['tentativas'])}")
    print("=" * 78)
    t = res["trilha_final"]
    if not t or not t["ordem"]:
        print("(sem trilha valida)")
        return
    print("ORDEM DE ESTUDO FINAL (* = escolhido pelo Qwen):")
    for i, nome in enumerate(t["ordem"], 1):
        print(f"  {i:2d}. {'*' if nome in t['selecionados'] else ' '} {nome}  [{fmt_ref(t['meta'][nome]['book_ref'])}]")


## Celula 22 -- Ground truth do colega e avaliação

Carrega os exemplos verificados no banco e mede F1 (antes/depois da correção) e a concordância do painel com o ground truth.

In [ ]:
# ==========================================================
# GROUND TRUTH (colega) + AVALIACAO
# Formato esperado de cada exemplo:
#   {"problema": "...", "topicos": ["Topico A", "Topico B"]}   <- tópicos NUCLEO (o que o Qwen deveria escolher)
# ou, se o colega entregar a trilha completa (com pre-requisitos):
#   {"problema": "...", "trilha": ["Pre-req 1", ..., "Topico A"]}
# ==========================================================
import pandas as pd


def nucleo_da_trilha(topicos):
    """Trilha completa -> topicos 'nucleo' = os que nao sao pre-requisito de outro
    topico da propria trilha (folhas do subgrafo induzido). Heuristica: se algum
    topico escolhido tambem for pre-requisito de outro escolhido, ele sera perdido."""
    com_sucessor = {a["de"] for a in buscar_arestas(topicos)}
    return [t for t in topicos if t not in com_sucessor]


def _canonicos(nomes):
    canon = {t.strip().lower(): t for t in topic_list}
    ok = [canon[n.strip().lower()] for n in nomes if n.strip().lower() in canon]
    desconhecidos = [n for n in nomes if n.strip().lower() not in canon]
    return ok, desconhecidos


def normalizar_ground_truth(exemplos):
    saida = []
    for ex in exemplos:
        nomes = ex.get("topicos") or nucleo_da_trilha(_canonicos(ex["trilha"])[0])
        ok, desc = _canonicos(nomes)
        if desc:
            print(f"[aviso] topicos fora do grafo ignorados: {desc}")
        saida.append({"problema": ex["problema"], "topicos": ok})
    return saida


def carregar_ground_truth(banco, exemplos):
    """Insere os exemplos verificados no banco (prioridade sobre os dos juizes)."""
    n = 0
    for ex in normalizar_ground_truth(exemplos):
        n += bool(banco.adicionar({"id": id_problema(ex["problema"]), "problema": ex["problema"],
                                   "tipo": "bom", "fonte": "ground_truth",
                                   "topicos": ex["topicos"], "confianca": 1.0}))
    print(f"{n} exemplos de ground truth no banco -> {banco.resumo()}")


def f1_conjuntos(pred, ref):
    pred, ref = set(pred), set(ref)
    if not pred or not ref:
        return 0.0
    tp = len(pred & ref)
    p, r = tp / len(pred), tp / len(ref)
    return 0.0 if tp == 0 else 2 * p * r / (p + r)


def avaliar_vs_ground_truth(exemplos, banco=None, max_rodadas=2, limiar_f1=0.8):
    """Roda o pipeline em cada exemplo (SEM registrar no banco) e compara com o GT.
    Compara 3 coisas: F1 da 1a tentativa, F1 apos a correcao e concordancia do
    painel com o GT (painel diz CORRETA <-> F1 >= limiar_f1)."""
    linhas = []
    for ex in normalizar_ground_truth(exemplos):
        res = pipeline_autocorrecao(ex["problema"], max_rodadas=max_rodadas, banco=banco,
                                    registrar=False, verbose=False)
        t0 = res["tentativas"][0]
        f1_ini = f1_conjuntos(t0["trilha"]["selecionados"], ex["topicos"])
        f1_fim = f1_conjuntos(res["trilha_final"]["selecionados"], ex["topicos"])
        linhas.append({"id": id_problema(ex["problema"]), "status": res["status"],
                       "painel_rodada0": t0["painel"]["veredito"],
                       "f1_inicial": f1_ini, "f1_final": f1_fim})
    df = pd.DataFrame(linhas)
    decisivos = df[df["painel_rodada0"].isin(["CORRETA", "INCORRETA"])]
    concord = ((decisivos["painel_rodada0"] == "CORRETA") == (decisivos["f1_inicial"] >= limiar_f1)).mean() \
        if len(decisivos) else float("nan")
    print(df.to_string(index=False))
    print(f"\nF1 medio: inicial={df['f1_inicial'].mean():.3f}  final={df['f1_final'].mean():.3f}")
    print(f"Concordancia painel x ground truth (F1>={limiar_f1}): {concord:.1%} "
          f"em {len(decisivos)}/{len(df)} casos decididos")
    return df


## Celula 23 -- Execução

Roda os dois pipelines sobre o `problem_text` da Célula 14.

In [ ]:
# ==========================================================
# EXECUCAO -- Pipeline 1 (auto-correcao) + Pipeline 2 (reforco de prompt)
# ==========================================================
banco = BancoExemplos()          # carrega /kaggle/working/banco_exemplos.json, se existir
print("Banco:", banco.resumo())

# Quando o colega entregar o ground truth:
# gt = [{"problema": "...", "topicos": ["...", "..."]}, ...]
# carregar_ground_truth(banco, gt)

resultado = pipeline_autocorrecao(problem_text, max_rodadas=2, banco=banco)
imprimir_resultado(resultado)

# Avaliacao (precisa de ground truth; o proprio problema e excluido dos exemplos do prompt):
# df_com  = avaliar_vs_ground_truth(gt, banco=banco)   # com exemplos fixos
# df_sem  = avaliar_vs_ground_truth(gt, banco=None)    # baseline, sem exemplos


In [10]:
import os
from datetime import datetime

# Pasta base para salvar os resultados
base_dir = "/kaggle/working/baseline_results"
os.makedirs(base_dir, exist_ok=True)

# Identificador do modelo (ajuste conforme o modelo carregado)
model_name = "qwen3-4b-base"   # mude para "qwen3-4b-v1" ou "qwen3-4b-v2" conforme necessário
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Nome da subpasta para este experimento
experiment_dir = os.path.join(base_dir, f"{model_name}_{timestamp}")
os.makedirs(experiment_dir, exist_ok=True)

# Para cada resposta, cria um arquivo .txt
for label, answer in results.items():
    # Nome do arquivo com espaços substituídos por underscore
    filename = label.replace(" ", "_").lower() + ".txt"
    filepath = os.path.join(experiment_dir, filename)

    # Conteúdo do arquivo: inclui o prompt e a resposta
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(f"Modelo: {model_name}\n")
        f.write(f"Data: {timestamp}\n")
        f.write(f"Tarefa: {label}\n")
        f.write("="*60 + "\n")
        f.write("PROMPT UTILIZADO:\n")
        f.write(prompts[label] + "\n\n")
        f.write("="*60 + "\n")
        f.write("RESPOSTA DO MODELO:\n")
        f.write(answer + "\n")

print(f"Resultados salvos em: {experiment_dir}")

NameError: name 'results' is not defined